In [1]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
import logging
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def generate_punchlines_finetuned_mistral_formatted(
    adapter_path_or_id,
    base_model_id="mistralai/Mistral-7B-v0.1",
    output_csv_path="finetuned_peft_mistral_punchlines.csv",
    input_data_path="data/test_data2.csv",
    seed=42,
    batch_size=8,
    max_new_tokens=50,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    hf_auth_token=None,
    use_quantization=False
):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logging.info(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(
        base_model_id,
        padding_side="left",
        use_auth_token=hf_auth_token,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    ) if use_quantization and torch.cuda.is_available() else None

    model_dtype = None if use_quantization else (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        device_map="auto",
        torch_dtype=model_dtype,
        quantization_config=quant_config,
        use_auth_token=hf_auth_token,
        trust_remote_code=True
    )

    model = PeftModel.from_pretrained(
        base_model,
        adapter_path_or_id,
        device_map="auto",
        use_auth_token=hf_auth_token
    ).eval()

    model.config.pad_token_id = tokenizer.pad_token_id

    # Load data
    data = pd.read_csv(input_data_path)
    setups = data['setup'].tolist()
    ground_truths = data['punchline'].tolist() if 'punchline' in data.columns else [None] * len(setups)

    results = []

    for i in tqdm(range(0, len(setups), batch_size), desc="Generating Punchlines"):
        batch_setups = setups[i:i + batch_size]
        batch_truths = ground_truths[i:i + batch_size]

        # --- FORMATTED PROMPTS ---
        prompts = [f"Setup: {s}\nPunchline:" for s in batch_setups]

        encoded = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors='pt'
        ).to(device)

        try:
            with torch.no_grad():
                outputs = model.generate(
                    input_ids=encoded["input_ids"],
                    attention_mask=encoded["attention_mask"],
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    top_k=top_k,
                    top_p=top_p,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    do_sample=True,
                    num_return_sequences=1
                )
        except Exception as e:
            logging.error(f"Generation error at batch {i}: {e}")
            for s, gt in zip(batch_setups, batch_truths):
                results.append({
                    'setup': s,
                    'ground_truth_punchline': gt,
                    'generated_punchline': 'GENERATION_ERROR'
                })
            continue

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        cleaned = [
            full[len(prompt):].strip()
            for full, prompt in zip(decoded, prompts)
        ]

        for s, gt, gen in zip(batch_setups, batch_truths, cleaned):
            results.append({
                'setup': s,
                'ground_truth_punchline': gt,
                'generated_punchline': gen
            })

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    result_df = pd.DataFrame(results)
    result_df.to_csv("temp.csv", index=False)
    # --- MODIFICATION ---
    output_dir = os.path.dirname(output_csv_path)
    if output_dir: # Only create directories if a path is specified
        os.makedirs(output_dir, exist_ok=True)
    # --- END MODIFICATION ---

    result_df.to_csv(output_csv_path, index=False)
    logging.info(f"Saved formatted punchline outputs to {output_csv_path}")

In [ ]:
# --- !!! Configure these paths !!! ---
ADAPTER_ID = "../models/mistral-7b-finetuned-filtered" # Your PEFT adapter ID/path
BASE_MODEL_ID = "mistralai/Mistral-7B-v0.1" # The base model used
INPUT_CSV = "../data/filtered_new_train_data_20k.csv"
OUTPUT_CSV = "filtered_train_data_finetuned_mistral_generated_punchlines.csv" # Example output path
# --- Optional ---
YOUR_HF_TOKEN = "" # Set your Hugging Face token if needed for private models/adapters
USE_QUANT = True # Set to True to try loading base model in 4-bit

# Basic check for input file
if not os.path.exists(INPUT_CSV):
    logging.error(f"Input file not found: {INPUT_CSV}. Please create it or change the path.")
else:
    logging.info("Starting punchline generation with the fine-tuned PEFT model...")
    generate_punchlines_finetuned_mistral_formatted(
        adapter_path_or_id=ADAPTER_ID,
        base_model_id=BASE_MODEL_ID,
        output_csv_path=OUTPUT_CSV,
        input_data_path=INPUT_CSV,
        batch_size=32,  # Adjust based on your GPU memory (start lower for PEFT)
        hf_auth_token=YOUR_HF_TOKEN,
        use_quantization=USE_QUANT # Pass the quantization flag
    )
    logging.info("Punchline generation finished.")

In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
import logging
import os

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def generate_punchlines_finetuned_llama_formatted(
    adapter_path_or_id,
    base_model_id,
    output_csv_path,
    input_data_path,
    batch_size=32,
    max_gen_length=50,
    temperature=0.8,
    top_p=0.9,
    top_k=50,
    seed=42,
    hf_auth_token=None,
    use_quantization=True
):
    """
    Generate punchlines using a fine-tuned PEFT model (LoRA adapters).
    
    Args:
        adapter_path_or_id: Path or HF Hub ID to the PEFT adapter
        base_model_id: Base model identifier (e.g., "meta-llama/Meta-Llama-3-8B")
        output_csv_path: Path to save the generated results
        input_data_path: Path to input CSV file with 'setup' column
        batch_size: Number of samples to process at once
        max_gen_length: Maximum number of tokens to generate
        temperature: Sampling temperature
        top_p: Nucleus sampling parameter
        top_k: Top-k sampling parameter
        seed: Random seed for reproducibility
        hf_auth_token: Hugging Face authentication token
        use_quantization: Whether to use 4-bit quantization
    """
    
    # Set seed for reproducibility
    set_seed(seed)
    
    logging.info(f"Loading base model: {base_model_id}")
    
    # Configure quantization if requested
    if use_quantization:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        
        # Load base model with quantization
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            token=hf_auth_token
        )
    else:
        # Load base model without quantization
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            token=hf_auth_token
        )
    
    logging.info(f"Loading PEFT adapter from: {adapter_path_or_id}")
    
    # Load PEFT model
    model = PeftModel.from_pretrained(base_model, adapter_path_or_id)
    model.eval()
    
    # Load tokenizer
    logging.info("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        adapter_path_or_id if os.path.exists(adapter_path_or_id) else base_model_id,
        trust_remote_code=True,
        token=hf_auth_token
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # For batched generation
    
    # Load input data
    logging.info(f"Loading input data from: {input_data_path}")
    df = pd.read_csv(input_data_path)
    
    if 'setup' not in df.columns:
        raise ValueError("Input CSV must contain a 'setup' column")
    
    setups = df['setup'].tolist()
    generated_punchlines = []
    
    logging.info(f"Generating punchlines for {len(setups)} setups...")
    
    # Process in batches
    for i in tqdm(range(0, len(setups), batch_size), desc="Generating"):
        batch_setups = setups[i:i + batch_size]
        
        # Create prompts in the same format as training
        prompts = [f"Setup: {setup}\nPunchline:" for setup in batch_setups]
        
        # Tokenize batch
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_gen_length,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                num_return_sequences=1
            )
        
        # Decode outputs
        for j, output in enumerate(outputs):
            generated_text = tokenizer.decode(output, skip_special_tokens=True)
            
            # Extract punchline (remove the prompt part)
            if "Punchline:" in generated_text:
                punchline = generated_text.split("Punchline:")[-1].strip()
            else:
                # Fallback: remove prompt length
                punchline = generated_text[len(prompts[j]):].strip()
            
            # Clean up punchline (remove any trailing newlines or extra text)
            punchline = punchline.split('\n')[0].strip()
            
            generated_punchlines.append(punchline)
    
    # Create output dataframe
    output_df = df.copy()
    output_df['generated_punchline'] = generated_punchlines
    output_df['seed'] = seed
    output_df['temperature'] = temperature
    output_df['top_p'] = top_p
    output_df['top_k'] = top_k
    
    # Save results
    os.makedirs(os.path.dirname(output_csv_path) if os.path.dirname(output_csv_path) else '.', exist_ok=True)
    output_df.to_csv(output_csv_path, index=False)
    logging.info(f"Results saved to: {output_csv_path}")
    
    return output_df


# --- !!! Configure these paths !!! ---
ADAPTER_ID = "../humor_gen_oct2024/1357629/llama-8b-finetuned-filtered"  # Your PEFT adapter ID/path
BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B"  # The base model used
INPUT_CSV = "../data/filtered_new_train_data_20k.csv"
OUTPUT_CSV = "filtered_train_data_finetuned_llama_generated_punchlines.csv"  # Example output path

# --- Optional ---
YOUR_HF_TOKEN = ""  # Set your Hugging Face token if needed
USE_QUANT = True  # Set to True to try loading base model in 4-bit

# Basic check for input file
if not os.path.exists(INPUT_CSV):
    logging.error(f"Input file not found: {INPUT_CSV}. Please create it or change the path.")
else:
    logging.info("Starting punchline generation with the fine-tuned PEFT model...")
    generate_punchlines_finetuned_llama_formatted(
        adapter_path_or_id=ADAPTER_ID,
        base_model_id=BASE_MODEL_ID,
        output_csv_path=OUTPUT_CSV,
        input_data_path=INPUT_CSV,
        batch_size=32,  # Adjust based on your GPU memory
        hf_auth_token=YOUR_HF_TOKEN,
        use_quantization=USE_QUANT
    )
    logging.info("Punchline generation finished.")

In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
import logging
import os


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def generate_punchlines_finetuned_qwen_formatted(
    adapter_path_or_id,
    base_model_id,
    output_csv_path,
    input_data_path,
    batch_size=32,
    max_gen_length=50,
    temperature=0.8,
    top_p=0.9,
    top_k=50,
    seed=42,
    hf_auth_token=None,
    use_quantization=True
):
    """
    Generate punchlines using a fine-tuned PEFT Qwen model (LoRA adapters).
    
    Args:
        adapter_path_or_id: Path or HF Hub ID to the PEFT adapter
        base_model_id: Base model identifier (e.g., "Qwen/Qwen3-8B")
        output_csv_path: Path to save the generated results
        input_data_path: Path to input CSV file with 'setup' column
        batch_size: Number of samples to process at once
        max_gen_length: Maximum number of tokens to generate
        temperature: Sampling temperature
        top_p: Nucleus sampling parameter
        top_k: Top-k sampling parameter
        seed: Random seed for reproducibility
        hf_auth_token: Hugging Face authentication token
        use_quantization: Whether to use 4-bit quantization
    """
    
   
    set_seed(seed)
    
    logging.info(f"Loading base model: {base_model_id}")
    
    
    if use_quantization:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        
        
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            token=hf_auth_token
        )
    else:
        
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            token=hf_auth_token
        )
    
    logging.info(f"Loading PEFT adapter from: {adapter_path_or_id}")
    
 
    model = PeftModel.from_pretrained(base_model, adapter_path_or_id)
    model.eval()
    
 
    logging.info("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        adapter_path_or_id if os.path.exists(adapter_path_or_id) else base_model_id,
        trust_remote_code=True,
        token=hf_auth_token
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  
    
   
    logging.info(f"Loading input data from: {input_data_path}")
    df = pd.read_csv(input_data_path)
    
    if 'setup' not in df.columns:
        raise ValueError("Input CSV must contain a 'setup' column")
    
    setups = df['setup'].tolist()
    generated_punchlines = []
    
    logging.info(f"Generating punchlines for {len(setups)} setups...")
    
    
    for i in tqdm(range(0, len(setups), batch_size), desc="Generating"):
        batch_setups = setups[i:i + batch_size]
        
        # Create prompts in the same format as training
        prompts = [f"Setup: {setup}\nPunchline:" for setup in batch_setups]
        
        
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
      
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_gen_length,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                num_return_sequences=1
            )
        
        
        for j, output in enumerate(outputs):
            generated_text = tokenizer.decode(output, skip_special_tokens=True)
            
            
            if "Punchline:" in generated_text:
                punchline = generated_text.split("Punchline:")[-1].strip()
            else:
                
                punchline = generated_text[len(prompts[j]):].strip()
            
           
            punchline = punchline.split('\n')[0].strip()
            
            generated_punchlines.append(punchline)
    
   
    output_df = df.copy()
    output_df['generated_punchline'] = generated_punchlines
    output_df['seed'] = seed
    output_df['temperature'] = temperature
    output_df['top_p'] = top_p
    output_df['top_k'] = top_k
    
   
    os.makedirs(os.path.dirname(output_csv_path) if os.path.dirname(output_csv_path) else '.', exist_ok=True)
    output_df.to_csv(output_csv_path, index=False)
    logging.info(f"Results saved to: {output_csv_path}")
    
    return output_df



ADAPTER_ID = "../humor_gen_oct2024/1549638/qwen_finetuned_unfiltred"  
BASE_MODEL_ID = "Qwen/Qwen3-8B"  
INPUT_CSV = "../humor_gen_oct2024/data/train_set_20k.csv"
OUTPUT_CSV = "filtered_train_data_finetuned_qwen_generated_punchlines.csv"  


YOUR_HF_TOKEN = ""  
USE_QUANT = True  


if not os.path.exists(INPUT_CSV):
    logging.error(f"Input file not found: {INPUT_CSV}. Please create it or change the path.")
else:
    logging.info("Starting punchline generation with the fine-tuned PEFT Qwen model...")
    generate_punchlines_finetuned_qwen_formatted(
        adapter_path_or_id=ADAPTER_ID,
        base_model_id=BASE_MODEL_ID,
        output_csv_path=OUTPUT_CSV,
        input_data_path=INPUT_CSV,
        batch_size=32,  
        hf_auth_token=YOUR_HF_TOKEN,
        use_quantization=USE_QUANT
    )
    logging.info("Punchline generation finished.")

In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
import logging
import os

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def generate_punchlines_finetuned_llama_formatted(
    adapter_path_or_id,
    base_model_id,
    output_csv_path,
    input_data_path,
    batch_size=32,
    max_gen_length=50,
    temperature=0.8,
    top_p=0.9,
    top_k=50,
    seed=42,
    hf_auth_token=None,
    use_quantization=True
):
    """
    Generate punchlines using a fine-tuned PEFT model (LoRA adapters).
    
    Args:
        adapter_path_or_id: Path or HF Hub ID to the PEFT adapter
        base_model_id: Base model identifier (e.g., "meta-llama/Meta-Llama-3-8B")
        output_csv_path: Path to save the generated results
        input_data_path: Path to input CSV file with 'setup' column
        batch_size: Number of samples to process at once
        max_gen_length: Maximum number of tokens to generate
        temperature: Sampling temperature
        top_p: Nucleus sampling parameter
        top_k: Top-k sampling parameter
        seed: Random seed for reproducibility
        hf_auth_token: Hugging Face authentication token
        use_quantization: Whether to use 4-bit quantization
    """
    
    # Set seed for reproducibility
    set_seed(seed)
    
    logging.info(f"Loading base model: {base_model_id}")
    
    # Configure quantization if requested
    if use_quantization:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        
        # Load base model with quantization
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            token=hf_auth_token
        )
    else:
        # Load base model without quantization
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            token=hf_auth_token
        )
    
    logging.info(f"Loading PEFT adapter from: {adapter_path_or_id}")
    
    # Load PEFT model
    model = PeftModel.from_pretrained(base_model, adapter_path_or_id)
    model.eval()
    
    # Load tokenizer
    logging.info("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        adapter_path_or_id if os.path.exists(adapter_path_or_id) else base_model_id,
        trust_remote_code=True,
        token=hf_auth_token
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # For batched generation
    
    # Load input data
    logging.info(f"Loading input data from: {input_data_path}")
    df = pd.read_csv(input_data_path)
    
    if 'setup' not in df.columns:
        raise ValueError("Input CSV must contain a 'setup' column")
    
    setups = df['setup'].tolist()
    generated_punchlines = []
    
    logging.info(f"Generating punchlines for {len(setups)} setups...")
    
    # Process in batches
    for i in tqdm(range(0, len(setups), batch_size), desc="Generating"):
        batch_setups = setups[i:i + batch_size]
        
        # Create prompts in the same format as training
        prompts = [f"Setup: {setup}\nPunchline:" for setup in batch_setups]
        
        # Tokenize batch
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_gen_length,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                num_return_sequences=1
            )
        
        # Decode outputs
        for j, output in enumerate(outputs):
            generated_text = tokenizer.decode(output, skip_special_tokens=True)
            
            # Extract punchline (remove the prompt part)
            if "Punchline:" in generated_text:
                punchline = generated_text.split("Punchline:")[-1].strip()
            else:
                # Fallback: remove prompt length
                punchline = generated_text[len(prompts[j]):].strip()
            
            # Clean up punchline (remove any trailing newlines or extra text)
            punchline = punchline.split('\n')[0].strip()
            
            generated_punchlines.append(punchline)
    
    # Create output dataframe
    output_df = df.copy()
    output_df['generated_punchline'] = generated_punchlines
    output_df['seed'] = seed
    output_df['temperature'] = temperature
    output_df['top_p'] = top_p
    output_df['top_k'] = top_k
    
    # Save results
    os.makedirs(os.path.dirname(output_csv_path) if os.path.dirname(output_csv_path) else '.', exist_ok=True)
    output_df.to_csv(output_csv_path, index=False)
    logging.info(f"Results saved to: {output_csv_path}")
    
    return output_df


# --- !!! Configure these paths !!! ---
ADAPTER_ID = "/humor_gen_oct2024/1549673/llama_finetuned_unfiltred"  # Your PEFT adapter ID/path
BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B"  # The base model used
INPUT_CSV = "/humor_gen_oct2024/data/train_set_20k.csv"
OUTPUT_CSV = "filtred_train_data_finetuned_llama_generated_punchlines.csv"  # Example output path

# --- Optional ---
YOUR_HF_TOKEN = ""  # Set your Hugging Face token if needed
USE_QUANT = True  # Set to True to try loading base model in 4-bit

# Basic check for input file
if not os.path.exists(INPUT_CSV):
    logging.error(f"Input file not found: {INPUT_CSV}. Please create it or change the path.")
else:
    logging.info("Starting punchline generation with the fine-tuned PEFT model...")
    generate_punchlines_finetuned_llama_formatted(
        adapter_path_or_id=ADAPTER_ID,
        base_model_id=BASE_MODEL_ID,
        output_csv_path=OUTPUT_CSV,
        input_data_path=INPUT_CSV,
        batch_size=32,  # Adjust based on your GPU memory
        hf_auth_token=YOUR_HF_TOKEN,
        use_quantization=USE_QUANT
    )
    logging.info("Punchline generation finished.")